<h1>Face Detetion with Convolutional Neural Network</h1>

<h2>Introduction</h2>

<h3>Dataset</h3>

<h2>Environment Set-Up</h2>

In [ ]:
import os
import sys
# Setting Hadoop home directory for the JVM
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-21.0.12"
os.environ["SPARK_HOME"] = r"C:\tools\Anaconda3\Lib\site-packages\pyspark"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = (
    os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep +
    os.path.join(os.environ["HADOOP_HOME"], "bin") + os.pathsep + 
    os.environ["PATH"]
    )

# TenserFlow / oneDNN CPU tuning
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["OMP_NUM_THREADS"] = "8"
os.environ["KMP_BLOCKTIME"] = "1"
os.environ["KMP_AFFINITY"] = "granularity=fine,compact,1,0"

In [ ]:
from PIL import Image

In [ ]:
import findspark
findspark.init()
import pyspark
from pyspark.ml.feature import Tokenizer, CountVectorizer, StringIndexer
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, IntegerType
from pyspark.sql.window import Window
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql import SparkSession

spark =( SparkSession.builder
        .master("local[*]")
        .config('spark.hadoop.home.dir', r'C:\hadoop')
        .config('spark.jars.packages','org.postgresql:postgresql:42.7.12')
        .getOrCreate()
)
print('Spark Version:', spark.version)
print('Hadoop Home:', os.environ["HADOOP_HOME"])

In [ ]:
BASE = ""
IMG_TRAIN = os.path.join(BASE, "images", "train")
IMG_VAL = os.path.join(BASE, "images", "val")
LBL_TRAIN = os.path.join(BASE, "labels", "train")
LBL_VAL = os.path.join(BASE, "labels", "val")
LBL2_DIR = os.path.join(BASE, "labels2")

<h2>Building PySpark DataFrame of the Annotations<h2>

In [ ]:
# Parsing YOLO labels files into a Spark Dataframe
schema = StructType([
    StructField('filename',StringType(), False),
    StructField('split',StringType(), False),
    StructField('cls',IntegerType(), True),
    StructField('x_center',DoubleType(), True),
    StructField('y_center',DoubleType(), True),
    StructField('width',DoubleType(), True),
    StructField('height',DoubleType(), True),
])

def parse_label_file(file_path, split):
    fname = os.path.splittext(os.path.basename(path))[0]
    rows = []
    with open(path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                rows.append((fname, split,
                             int(float(parts[0])),
                             float(parts[1]), float(parts[2]),
                             float(parts[3]), float(parts[4])))
        elif len(parts) == 0:
        #malformed line : records with null so its visible in the audit
        rows.append((fname, split, None, None, None, None, None))
    return rows

label_rows = []
for split, path in [('train', LBL_TRAIN), ]:
    for p in glob.glob(os.path.join(path, '*.txt')):
        label_rows.extend(parse_label_file(p, split))
labels_df = spark.createDataFrame(label_rows, schema=schema).cache()
print('Total annotated boxes: ', labels_df.count())
labels_df.show(5, truncate=False)

<h2>Exploratory Data Analysis</h2>

<h3>Data Quality Assessment<h3>

In [ ]:
# Completenes
# Will check orphaned images and orphaned label files
def list_images(folder, split):
    return [(os.path.splitext(os.path.basename(p))[0], split)
            for p in glob.glob(os.path.join(folder, '*.jpg'))]

img_rows = (list_images(IMG_TRAIN,'train') + list_images(IMG_VAL, 'val'))

img_schema = StructType([StructuredField('filename',StringType(), False), StructureField('split',StringType())])

images_df = spark.createDataFrame(img_rows, schema=img_schema).cache()

label_names_df = (labels_df.select('filename','split').distinct())

orphaned_images = images_df.join(label_names_df, on = ['filename','split'], how='left_anti')

orphan_labels = label_names_df.join(images_df, on = ['filename','split'], how='left_anti')

In [ ]:
print('Images without labels:', orphaned_images.count())
print('Labels without images:', orphaned_labels.count())
orphaned_images.show(5, truncate=False)

In [ ]:
# Self consistency
#Comparing labels/YOLO vs labels2
def parse_pixel_label(path):
    #return list of (x_min, y_min, x_max, y_max) in pixels
    boxes = []
    if not os.path.exists(path):
        return boxes
    with open(path,'r') as f:
        for line in f:
            parts = line.strip().split()
            parts = line.strip().split()
            # Format: Human face x_min, y_min, x_max, y_max
            if len(parts) >= 6 and parts[0].lower() == 'human' and parts[1].lower() == 'face':
                boxes.append(tuple(float(v) for v in parts[2:6]))
    return boxes

consistency_report = []
all_label_names = [r[0] + '.txt' for r in label_names_df.select('filename').collect()]

#Selecting 1000 files to check for consistency
random.seed(42)
sample_names = random.sample(all_label_names, min(1000, len(all_label_names)))
for name in sample_names:
    stem = os.path.splitext(name)[0]
    #Locate image
    img_path = None
    for d in (IMG_TRAIN, IMG_VAL):
        cand = os.path.join(d, stem +'.jpg')
        if os.path.exists(cand):
            img_path = cand
            break
    if img_path is None:
        continue
    with Image.open(img_path) as im:
        iw, ih = im.size
    # YOLO boxes
    yolo_path = os.path.join(LBL_TRAIN if 'train' in img_path else LBL_VAL, name)
    
    yolo_boxes = []
    with open(yolo_path, 'r') as f:
        for line in f:
            p = line.strip().split()
            if len(p) == 5:
                _, xc, yc, w, h = map(float, p)
                yolo_boxes.append(((xc - w/2)*iw, (yc- h/2)*ih,
                                   (xc + w/2)*iw, (yc + h/2 )*ih))
                
    pix_boxes = parse_pixel_label(os.path.join(LBL2_DIR, name))
    if len(yolo_boxes) != len(pix_boxes) and len(pix_boxes) > 0:
        diffs = [max(abs(a-b) for a,b in zip(y,p)) for y,p in zip(yolo_boxes, pix_boxes)]
        consistency_report.append((stem, len(pix_boxes), float(np.mean(diffs)),float(np.max(diffs))))
    else:
        consistency_report.append((stem, len(pix_boxes), np.nan, np.nan))


In [ ]:
cons_df = spark.createDataFrame(consistency_report, schema=StructType([
    StructField('filename',StringType()),
    StructField('n_boxes',IntegerType()),
    StructField('mean_px_diff',DoubleType()),
    StructField('max_px_diff',DoubleType())
]))
print('Consistency Summary (pixels): ')
cons_df.select(F.mean('mean_px_diff').alias('avg_mean_px_diff'),
               F.max('max_px_diff').alias('worst_max_diff')).show()

In [ ]:
# Box size and aspect ratio distribution
# For reportign I will need img dimensions, so I will build a small image-metadate table
def get_dims(row):
    for d in (IMG_TRAIN, IMG_VAL):
        p = os.path.join(d, row['filename'] + '.jpg')
        if os.path.exists(p):
            with Image.open(p) as im:
                return (row['filename'], row['split'], int(im.size[0]), int(im.size[1]))
    return (row['filename'], row['split'], None, None)

dims_rows = [get_dims(r.asDict()) for r in image_df.collect()]
dims_df = spark.createDataFrame(dims_rows, schema=StructType([
    StructField('filename',StringType()),
    StructField('split',StringType()),
    StructField('imag_width',IntegerType()),
    StructField('img_height',IntegerType())
])).cache()

boxes_df = (labels_df
            .join(dims_df, on=['filename','split'], how='inner')
            .withColumn('box_w_px', F.col('w')* F.col('img_w'))
            .withColumn('box_h_px', F.col('h') * F.col('img_h'))
            .withColumn('box_area_px', F.col('box_w_px') * F.col('box_h_px'))
            .withColumn('aspect_ratio', F.col('box_w_px') / F.col('box_h'))
            )

boxes_df.select(
    F.count('*').alias('n_boxes'),
    F.round(F.mean('box_area_px'), 1).alias('mean_area_px'),
    F.round(F.expr('percentile_approx(box_area_px, 0.5)'),1).alias('median_area_px'),
    F.round(F.expr('percentile_approx(box_area_px, 0.05)'),1).alias('p05_area_px'),
    F.round(F.expr('aspect_ration'),3).alias('mean_aspect'),
    F.min('box_area_px').alias('min_area_px'),
    F.max('box_area_px').alias('max_area_px')
).show()


In [ ]:
# Count of tiny boxes (below 16x16 px)
tiny_boxes = boxes_df.filter(F.col('box_area_px') < 256).count()
print(f'Boxes smaller than 16x16: {tiny_boxes} ({tiny_boxes/(boxes_df.count()*100):.2f}%)')

In [ ]:
# Negatives: Images with zero boxes
labelled = labels_df.select('filename', 'split').distinct()
negatives = images_df.join(labelled, on=['filename','split'], how='left_anti')
print('Images with no faces (negatives): ', negatives.count())

In [ ]:
# Near-duplicate detection
def ahash(path, hash_size=8):
    with Image.open(path) as im:
        im = im.convert('L').resize((hash_size, hash_size))
        pixels = np.array(im, dtype=np.float32)
        return int(''.join((pixels > pixels.mean()).astype(int).flatten().astype(str)),2)
hash_rows = []
for split, folder in [('train', IMG_TRAIN), ('val',IMG_VAL)]:
    for p in glob.glob(os.path.join(folder, '*.jpg')):
        hash_rows.append((os.path.splitext(os.path.basename(p))[0], split, ahash(p)))

hash_df = spark.createDataFrame(hash_rows, schema=StructType([StructField('filename',StringType()),
                                                                          StructField('split',StringType()),
                                                                          StructField('ahash',IntegerType())])).cache()
dup_pairs = (hash_df.alias('a').join(
    hash_df.alias('b'),
    (F.col('a.ahash') == F.col('b.ahash')) & (F.col('a.filename') < F.col('b.filename')))
             .select(F.col('a.filename').alias('file_a'), F.col('b.filename').alias('file_b'))
             )


In [ ]:
print('Exact-hash duplicates pairs:', dup_pairs.count())

<h3>Data Visualisation</h3>

In [ ]:
# Box area distribution
fig, axes = plt.subplots(2,2, figsize(14,10))

# 1.1 Box Area distribution (log10)
pdf = boxes_df.select('box_area_px').toPandas()
pdf['log_area'] = np.log10(pdf['box_area_px']).clip(lower=1)
sns.histplot(pdf['log_area'], bins=50, ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Distribution of Bouding-box area (log10 px^2)')
axes[0,0].set_xlabel('log10(area px^2)')

# 1.2 Aspect Ration
pdf_aspect = boxes_df.select('aspect_ration').toPandas()
sns.histplot(pdf_aspect['aspect_ration'].clip(0,3), bins=60, ax=axes[0,1], color='indianred')
axes[0,1].set_title('Bounding-box aspect ration (w/h)')
axes[0,1].set_xlabel('Aspect Ration')

# 2.1  Box area vs image area
# Checking if scale correlate with resolution
pdf_scale = boxes_df.select('img_w', 'img_h','box_area_px').toPandas()
pdf_scale['img_area'] = pdf_scale['img_w'] * pdf_scale['img_h']
axes[1,0].scatter(pdf_scale['img_area'], pdf_scale['box_area_px'], alpha=0.15, s=8, color='seagreen')
axes[1,0].set_xscale('log')
axes[1,0].set_yscale('log')
axes[1,0].set_title('Box area vs Image area')
axes[1,0].set_xlabel('Image Area (px^2)')
axes[1,0].set_ylabel('Box Area (px^2)')

# Boxes per Image
pdf_bpi = (labels_df.groupby('filename').count()
           .select('count').toPandas())
sns.histplot(pdf_bpi['count'].clip(0,20), bins=20, ax=axes[1,1], color='slategray')
axes[1,1].set_title('Faces per Image')
axes[1,1].set_xlabel('Number of Faces')

plt.tight_layout()
plt.show()

<h2>Feature Pre-processing</h2>

In [ ]:
# Cleaning and splitting
MIN_BOX_AREA_PX = 0
SEED = 42
random.seed(42)
# Building the the set of filenames to keep
keep_df = (boxes_df
           .filter(F.col('box_area_pc') >= MIN_BOX_AREA_PX)
           .select('filename','split')
           .distinct())
# Re-partition into 70/10/20
all_files = keep_df.toPandas().sample(frac=1, random_state=SEED).reset_idex(drop=True)
n = len(all_files)
n_train = int(0.70 * n)
n_val = int(0.10 * n)

all_files['new_split'] = 'train'
all_files.loc[n_train:n_train + n_val -1, 'new_split'] = 'val'
all_files.loc[n_train + n_val:, 'new_split'] = 'test'
print(all_files['new_split'].value_counts())

In [ ]:
# Creating the Ultralytics-compatible directory tree
YOLO_ROOT = os.path.join(BASE, 'yolo')
for sub in ('images/train', 'images/val','images/test','labels/train','labels/val','labels/test'):
    os.makedirs(os.path.join(YOLO_ROOT, sub), exist_ok=True)

def move_pair(stem, src_split, dst_split):
    # image
    src_img = os.path.join(IMG_TRAIN if src_split == 'train' else IMG_VAL, stem + '.jpg')
    dst_img = os.path.join(YOLO_ROOT, 'images', dst_split, stem + '.jpg')
    if os.path.exists(src_img):
        shutil.copy(src_img, dst_img)
    
    #label
    src_lbl = os.path.join(LBL_TRAIN if src_split == 'train' else LBL_VAL, stem + '.txt')
    dst_lbl = os.path.join(YOLO_ROOT, 'labels', dst_split, stem + '.txt')
    if os.path.exists(src_lbl):
        shutil.copy(src_img, dst_lbl)
    
for _, row in all_giles.iterrows():
    move_pair(row['filename'], row['split'], row['new_split'])

In [ ]:
# Write data.yaml
yaml_content = f"""path: {YOLO_ROOT}
train: images/train
val: images/val
test: images/test
names:
    0: face
"""

with open(os.path.join(YOLO_ROOT, 'data.yaml'), 'w') as f:
    f.write(yaml_content)
print(yaml_content)

<h2>Model Training </h2>

In [ ]:
model = YOLO('yolov8n.pt')
results = model.train(
    data=os.path.join(YOLO_ROOT, 'data.yaml'),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    project='runs',
    name='face_dectector_baseline',
    seed=SEED
    )

In [ ]:
# Capturing training log into a 'history' dictionary
results_csv = os.path.join('runs','face_detector_baseline','results.csv')
hist_df = pd.read_csv(results_csv)
hist_df.columns = [c.strip() for c in hist_df.columns]

history = {
    'epoch': hist_df['epoch'].values,
    'loss': hist_df['train/box_loss'].values + hist_df['train/cls_loss'].values + hist_df['train/dfl_loss'].values,
    'val_loss': hist_df['val/box_loss'].values + hist_df['val/cls_loss'].values + hist_df['val/dfl_loss'].values,
    'precision': hist_df['metrics/precision(B)'].values,
    'recall': hist_df['metrics/recall(B)'].values, 
    'mAP50': hist_df['metrics/mAP50(B)'].values, 
    'mAP50_95': hist_df['metrics/mAP50-95(B)'].values,
}

In [ ]:
# Learning-curve plots
fig, axes = plt.subplots(2,2, figsize(14,8))

axes[0, 0].plot(history['epoch'], history['loss'], label='train loss')
axes[0, 0].plot(history[epoch], history['val_loss'], label='val loss')
axes[0, 0].set_title('Total loss vs epoch')
axes[0, 0].legend()

axes[0, 1].plot(history['epoch'], history['mAP50'], label='mAP@0.5')
axes[0, 1].plot(history['epoch'], history['mAP50_95'], label='mAP@0.5:0.95')
axes[0, 1].set_title(mAP vs epoch)
axes[0,1].legend()

axes[1, 1].scatter(history['val_loss'], history['mAP50'], c = history['epoch'], cmap='viridis')
axes[1, 1].set_xlabel('val loss')
axes[1, 1].set_ylabel('mAP@0.5')
axes[1, 1].set_title('mAP@0.5 vs val loss (colour = epoch)')

plt.tight_layout()
plt.show()

In [ ]:
# Evaluation on test set
metrics = model.val(
    data=os.path.join(YOLO_ROOT, 'data.yaml'),
    split='test',
    imgsz=640,
    conf=0.25,
    iou=0.45,
)
print(f'mAP@0.5': {metrics.box.map50:.4f})
print(f'mAP@0.5:0.95': {metrics.box.map:.4f})
print(f'Precision': {metrics.box.mp:.4f})
print(f'Recall': {metrics.box.mr:.4f})

In [ ]:
# Inference Speed
speed = metrics.speed
fps = 1000.0 / speed['inference']
print(f'Inference FPS : ' {fps:.1f})